In [29]:
import argparse

import d3rlpy
dataset, env = d3rlpy.datasets.get_d4rl('halfcheetah-medium-expert-v2')



Compiling /home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/mujoco_py/cymj.pyx because it changed.
[1/1] Cythonizing /home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/mujoco_py/cymj.pyx


performance hint: /home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/mujoco_py/cymj.pyx:67:0: Exception check on 'c_warning_callback' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'c_warning_callback' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'c_warning_callback' to allow an error code to be returned.
performance hint: /home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/mujoco_py/cymj.pyx:104:0: Exception check on 'c_error_callback' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'c_error_callback' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'c_error_callback' to allow an error code to be returned.

Error compiling Cython file:
-------------------------------------

CompileError: /home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/mujoco_py/cymj.pyx

In [5]:
import pickle 
dataset = "hopper-medium-expert-v2"
env_name = dataset.split("-")[0][0].upper() + dataset.split("-")[0][1:] + "-v4"
print(env_name)

pkl_path = f"../../../../master_thesis/reproducing_decision_transformer/gymnasium/data/{dataset}.pkl"
with open(pkl_path, "rb") as f:
    episodes = pickle.load(f)

print(type(episodes))
print(len(episodes))
print(episodes[0].keys())
print(episodes[0]["observations"].shape)


Hopper-v4
<class 'list'>
3213
dict_keys(['observations', 'next_observations', 'actions', 'rewards', 'terminals'])
(470, 11)


In [6]:
import numpy as np
import pickle
from d3rlpy.dataset.components import Episode  # Ensure this imports the default Episode class

def load_and_convert_episodes(pkl_path):
    # Load your episodes from the pickle file
    with open(pkl_path, "rb") as f:
        raw_episodes = pickle.load(f)
    
    converted_episodes = []
    for ep in raw_episodes:
        # Assume each ep is a dict with keys as shown in your printout.
        observations = ep["observations"]           # shape (470, 11)
        actions = ep["actions"]
        rewards = ep["rewards"]
        terminals = ep["terminals"]
        if isinstance(terminals, (list, np.ndarray)):
            terminated = bool(terminals[-1])
        else:
            terminated = bool(terminals)
        # Optionally, next_observations is available as well.
        next_observations = ep.get("next_observations", None)
        
        # Create an Episode object.
        # The Episode class in d3rlpy typically accepts these arrays directly.
        episode = Episode(
            observations=observations,
            actions=actions,
            rewards=rewards,
            terminated=terminated,
        )
        converted_episodes.append(episode)
    
    return converted_episodes

# Example usage:
episodes = load_and_convert_episodes(pkl_path)
print(f"Converted {len(episodes)} episodes.")

# Now, to create a ReplayBuffer with these episodes:
from d3rlpy.dataset import ReplayBuffer, FIFOBuffer

buffer_impl = FIFOBuffer(limit=1000000)
replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=episodes)

Converted 3213 episodes.
2025-05-15 15:11.53 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[()])
2025-05-15 15:11.53 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-05-15 15:11.53 [info     ] Action size has been automatically determined. action_size=3


In [7]:
import numpy as np
import pickle
from d3rlpy.dataset.components import Episode

def convert_raw_episode(raw_ep):
    # Convert to NumPy arrays if they aren't already.
    observations = np.array(raw_ep["observations"])
    actions = np.array(raw_ep["actions"])
    rewards = np.array(raw_ep["rewards"])
    terminals = raw_ep["terminals"]

    # Make sure rewards is 2D: shape (T, 1) rather than (T,)
    if rewards.ndim == 1:
        rewards = rewards.reshape(-1, 1)
    
    # For the termination flag, take the last element of terminals.
    if isinstance(terminals, (list, np.ndarray)):
        terminated = bool(terminals[-1])
    else:
        terminated = bool(terminals)
    
    return Episode(
        observations=observations,
        actions=actions,
        rewards=rewards,
        terminated=terminated
    )

def load_and_convert_episodes(pkl_path):
    with open(pkl_path, "rb") as f:
        raw_episodes = pickle.load(f)
    return [convert_raw_episode(ep) for ep in raw_episodes]

# Example usage:
episodes = load_and_convert_episodes(pkl_path=pkl_path)
print(f"Loaded {len(episodes)} episodes.")

from d3rlpy.dataset import ReplayBuffer, FIFOBuffer

buffer_impl = FIFOBuffer(limit=1000000)
replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=episodes)

Loaded 3213 episodes.
2025-05-15 15:12.03 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-05-15 15:12.03 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-05-15 15:12.03 [info     ] Action size has been automatically determined. action_size=3


In [24]:
import numpy as np
import pickle
from d3rlpy.dataset.components import Episode

dataset = "halfcheetah-medium-expert-v2"
if "halfcheetah" in dataset:
    env_name = "HalfCheetah-v4"
elif "hopper" in dataset:
    env_name = "Hopper-v4"
elif "walker" in dataset:
    env_name = "Walker2d-v4"

#env_name = dataset.split("-")[0][0].upper() + dataset.split("-")[0][1:] + "-v5"
print(env_name)

pkl_path = f"../../../../master_thesis/reproducing_decision_transformer/gymnasium/data/{dataset}.pkl"

def convert_raw_episode(raw_ep):
    # Convert raw observations to a NumPy array and then to a list of individual observations.
    observations = np.array(raw_ep["observations"])

    # Ensure actions and rewards are NumPy arrays.
    actions = np.array(raw_ep["actions"])
    rewards = np.array(raw_ep["rewards"])
    # For rewards, ensure they have an extra dimension (T, 1)
    if rewards.ndim == 1:
        rewards = rewards.reshape(-1, 1)
    
    # Use the last element of "terminals" as the terminated flag.
    terminals = raw_ep["terminals"]
    if isinstance(terminals, (list, np.ndarray)):
        terminated = bool(terminals[-1])
    else:
        terminated = bool(terminals)
    
    return Episode(
        observations=observations,
        actions=actions,
        rewards=rewards,
        terminated=terminated
    )

def load_and_convert_episodes(pkl_path):
    with open(pkl_path, "rb") as f:
        raw_episodes = pickle.load(f)
    return [convert_raw_episode(ep) for ep in raw_episodes]

# Example usage:
episodes = load_and_convert_episodes(pkl_path=pkl_path)
print(f"Loaded {len(episodes)} episodes.")

from d3rlpy.dataset import ReplayBuffer, FIFOBuffer

buffer_impl = FIFOBuffer(limit=1000000)
replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=episodes)


HalfCheetah-v4
Loaded 2000 episodes.
2025-05-15 16:18.39 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(6,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(17,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-05-15 16:18.39 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-05-15 16:18.39 [info     ] Action size has been automatically determined. action_size=6


In [9]:
import gymnasium as gym
import d3rlpy
import argparse
# parser.add_argument("--dataset", type=str, default="hopper-medium-v0")
# parser.add_argument("--seed", type=int, default=1)
# parser.add_argument("--gpu", type=int)
# parser.add_argument("--compile", action="store_true")
# args = parser.parse_args()

args = argparse.Namespace()
args.dataset = dataset
args.seed = 1
args.gpu = 0
args.compile = False

import gym
env = gym.make(env_name)

#dataset, env = d3rlpy.datasets.get_dataset(args.dataset)

# fix seed
d3rlpy.seed(args.seed)
d3rlpy.envs.seed_env(env, args.seed)

if "halfcheetah" in args.dataset:
    target_return = 6000
elif "hopper" in args.dataset:
    target_return = 3600
elif "walker" in args.dataset:
    target_return = 5000
else:
    raise ValueError("unsupported dataset")

In [10]:
print(env.observation_space)

Box(-inf, inf, (17,), float64)


In [11]:
dt = d3rlpy.algos.DecisionTransformerConfig(
    batch_size=64,
    learning_rate=1e-4,
    optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=100#10000
        ),
    ),
    encoder_factory=d3rlpy.models.VectorEncoderFactory(
        [128],
        exclude_last_activation=True,
    ),
    observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
    reward_scaler=d3rlpy.preprocessing.MultiplyRewardScaler(0.001),
    position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
    context_size=20,
    num_heads=1,
    num_layers=3,
    max_timestep=1000,
    compile_graph=args.compile,
).create(device="cpu")

dt.fit(
    replay_buffer,
    n_steps=100,#100000,
    n_steps_per_epoch=10,#1000,
    save_interval=10,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"DT_{args.dataset}_{args.seed}",
)

2025-05-15 15:12.29 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(17,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(6,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=6)
2025-05-15 15:12.29 [debug    ] Fitting observation scaler...  observation_scaler=standard
2025-05-15 15:12.29 [debug    ] Building models...            
2025-05-15 15:12.30 [debug    ] Models have been built.       
2025-05-15 15:12.30 [info     ] Directory is created at d3rlpy_logs/DT_halfcheetah-medium-expert-v2_1_20250515151230
2025-05-15 15:12.30 [info     ] Parameters                     params={'observation_shape': [17], 'action_size': 6, 'config': {'type': 'decision_transformer', 'params': {'batch_size': 64, 'gamma': 0.99, 'observation_scaler': {'type': 'standard', 'params': {'mean': [-0.044943085813236805, 0.03229907396858086, 0

Epoch 1/10: 100%|██████████| 10/10 [00:02<00:00,  3.82it/s, loss=4.86]
/home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


2025-05-15 15:13.10 [info     ] DT_halfcheetah-medium-expert-v2_1_20250515151230: epoch=1 step=10 epoch=1 metrics={'time_sample_batch': 0.01418161392211914, 'time_algorithm_update': 0.2451770782470703, 'loss': 4.814155769348145, 'time_step': 0.25960373878479004, 'environment': -130.1477937744413} step=10


Epoch 2/10: 100%|██████████| 10/10 [00:02<00:00,  3.53it/s, loss=4.67]


2025-05-15 15:13.55 [info     ] DT_halfcheetah-medium-expert-v2_1_20250515151230: epoch=2 step=20 epoch=2 metrics={'time_sample_batch': 0.01510317325592041, 'time_algorithm_update': 0.2658839702606201, 'loss': 4.522355604171753, 'time_step': 0.28125600814819335, 'environment': -121.22646556502289} step=20


...

2025-05-15 15:17.45 [info     ] DT_halfcheetah-medium-expert-v2_1_20250515151230: epoch=8 step=80 epoch=8 metrics={'time_sample_batch': 0.013174200057983398, 'time_algorithm_update': 0.22135543823242188, 'loss': 0.8196473598480225, 'time_step': 0.23474409580230712, 'environment': -48.39019442050778} step=80


Epoch 9/10: 100%|██████████| 10/10 [00:02<00:00,  4.29it/s, loss=0.663]


2025-05-15 15:18.23 [info     ] DT_halfcheetah-medium-expert-v2_1_20250515151230: epoch=9 step=90 epoch=9 metrics={'time_sample_batch': 0.011873078346252442, 'time_algorithm_update': 0.219136643409729, 'loss': 0.6909113585948944, 'time_step': 0.23125877380371093, 'environment': -56.06577586451264} step=90


Epoch 10/10: 100%|██████████| 10/10 [00:02<00:00,  4.25it/s, loss=0.574]


2025-05-15 15:18.58 [info     ] DT_halfcheetah-medium-expert-v2_1_20250515151230: epoch=10 step=100 epoch=10 metrics={'time_sample_batch': 0.014368963241577149, 'time_algorithm_update': 0.21880598068237306, 'loss': 0.6053092956542969, 'time_step': 0.2333613872528076, 'environment': -67.51329893939302} step=100
2025-05-15 15:18.58 [info     ] Model parameters are saved to d3rlpy_logs/DT_halfcheetah-medium-expert-v2_1_20250515151230/model_100.d3


In [23]:
import gymnasium as gym
gym.make("Walker2d-v4")

<TimeLimit<OrderEnforcing<PassiveEnvChecker<Walker2dEnv<Walker2d-v4>>>>>

In [17]:
!pip list | grep gymnasium
!pip list | grep mujoco

gymnasium                1.0.0
gymnasium-robotics       1.3.1
mujoco                   2.3.3
mujoco-py                2.1.2.14
